In [ ]:
import numpy as np
import pandas as pd
import os
import kagglehub

In [ ]:
import os
os.makedirs("/kaggle/working/outputs/models", exist_ok=True)
print(sorted(os.listdir("/kaggle/working/outputs/models")))

In [ ]:
import subprocess, sys, importlib.util

def ensure_timm():
    """Install timm only if missing; retry on flaky network; don't hard-crash."""
    if importlib.util.find_spec("timm") is not None:
        return  # already installed (e.g. re-running in same session)
    for attempt in range(3):
        # no exact pin: 'timm==1.0.11' can fail to resolve while plain 'timm' works
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
        if importlib.util.find_spec("timm") is not None:
            return
        print(f"  timm install attempt {attempt+1} failed, retrying...")
    raise RuntimeError("Could not install timm — check Settings -> Internet is ON.")

ensure_timm()

import os, gc, glob, time, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
import timm
from PIL import Image
from tqdm import tqdm   # plain text bar (renders in notebooks AND terminals)
warnings.filterwarnings("ignore")

# ----- Reproducibility: seed every RNG -----
def seed_everything(seed: int = 42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything(42)

# ----- GPU + backend setup -----
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True            # fixed 224x224 input -> pure speed win
torch.backends.cuda.matmul.allow_tf32 = True     # no-op on T4, helps on A100/P100

# ----- Mixed precision (AMP) is enabled per-run via this flag -----
USE_AMP = torch.cuda.is_available()

print("PyTorch :", torch.__version__)
print("NumPy   :", np.__version__)
print("timm    :", timm.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("Memory  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: CPU only — enable T4 in Settings -> Accelerator.")
print("AMP     :", USE_AMP)
print("CELL 1 complete.")

In [ ]:

# CELL 2 — CONFIGURATION

CONFIG = {
    # paths (resolved robustly in Cell 3 if these don't exist)
    "metadata_csv": "/kaggle/input/datasets/sadi17/pad-ufes-20/metadata.csv",
    "images_root":  "/kaggle/input/datasets/sadi17/pad-ufes-20/images",
    "output_root":  "/kaggle/working/outputs",

    # data / training
    "image_size":  224,
    "num_classes": 6,
    "batch_size":  32,
    "epochs":      40,
    "warmup_epochs": 3,
    "patience":    8,
    "lr_backbone": 3e-5,
    "lr_head":     3e-4,
    "weight_decay": 1e-4,

    # cross-validation
    "n_folds":     5,
    "val_fraction": 0.15,    # of train, held out for early stopping (patient-grouped)
    "seed":        42,

    # runtime
    "num_workers": 2,
    "use_amp":     USE_AMP,
    "device":      DEVICE,
}


SUBDIRS = ["models", "metrics", "predictions", "tables", "figures",
           "logs", "gradcam", "metadata_analysis", "fold_results", "paper_outputs"]
PATHS = {}
for d in SUBDIRS:
    p = os.path.join(CONFIG["output_root"], d)
    os.makedirs(p, exist_ok=True)
    PATHS[d] = p


with open(os.path.join(CONFIG["output_root"], "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Output tree created under:", CONFIG["output_root"])
for d in SUBDIRS:
    print(f"  outputs/{d}/")
print("CELL 2 complete.")

In [ ]:

# CELL 3 — DATASET LOADING

meta_path = CONFIG["metadata_csv"]
if not os.path.exists(meta_path):
    meta_path = glob.glob("/kaggle/input/**/metadata.csv", recursive=True)[0]
images_root = CONFIG["images_root"]
if not os.path.isdir(images_root):
    images_root = glob.glob("/kaggle/input/**/images", recursive=True)[0]
print("metadata:", meta_path)
print("images  :", images_root)

df = pd.read_csv(meta_path)
print("Raw metadata shape:", df.shape)


path_index = {os.path.basename(p): p for p in
              glob.glob(os.path.join(images_root, "**", "*.png"), recursive=True)}
df["image_path"] = df["img_id"].map(path_index)
n_before = len(df)
df = df[df["image_path"].notna()].reset_index(drop=True)
print(f"Images matched on disk: {len(df)} (dropped {n_before - len(df)} unmatched)")


CLASS_NAMES = sorted(df["diagnostic"].unique().tolist())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
df["label"] = df["diagnostic"].map(CLASS_TO_IDX).astype(int)


df["_patient_id"] = df["patient_id"].values

df = df.drop(columns=[c for c in ["patient_id", "lesion_id", "img_id"] if c in df.columns])

# ----- dataset summary -----
print("\n========== DATASET SUMMARY ==========")
print("Images           :", len(df))
print("Patients         :", df["_patient_id"].nunique())
print("Classes          :", CLASS_TO_IDX)
print("Metadata columns :", [c for c in df.columns
                             if c not in ("image_path", "label", "_patient_id", "diagnostic")])
print("\nClass distribution (image-level):")
print(df["diagnostic"].value_counts().reindex(CLASS_NAMES).to_string())
print("\nMean images per patient:",
      round(len(df) / df["_patient_id"].nunique(), 2))
print("CELL 3 complete.")

In [ ]:
# =====================================================================
# CELL 4 — METADATA PROCESSING

CAT_FEATURES = ["smoke", "drink", "background_father", "background_mother", "pesticide",
                "gender", "skin_cancer_history", "cancer_history", "has_piped_water",
                "has_sewage_system", "fitspatrick", "region",
                "itch", "grew", "hurt", "changed", "bleed", "elevation"]
CONT_FEATURES = ["age", "diameter_1", "diameter_2"]


class MetaPreprocessor:
    """Fit on train rows; transform any row to a fixed-length float vector.
    include_biopsy: if True, append the biopsy flag as an extra binary feature
    (used ONLY by the leakage-demonstration experiment).
    levels: optional dict {feature: [values...]} giving a FIXED global
    vocabulary. If provided, used instead of re-deriving levels from `dft`
    (guarantees identical dimensionality across folds)."""
    def __init__(self, include_biopsy=False, levels=None):
        self.include_biopsy = include_biopsy
        self.fixed_levels = levels

    def fit(self, dft):
        self.levels, self.mode = {}, {}
        for f in CAT_FEATURES:
            obs = [str(v) for v in dft[f].dropna().tolist()]
            self.levels[f] = self.fixed_levels[f] if self.fixed_levels else sorted(set(obs))
            self.mode[f] = pd.Series(obs).mode()[0] if obs else "missing"
        self.cmean = {f: float(dft[f].dropna().mean() or 0.0) for f in CONT_FEATURES}
        self.cstd  = {f: (float(dft[f].dropna().std()) or 1.0) for f in CONT_FEATURES}
        self.feature_names = []
        for f in CAT_FEATURES:
            self.feature_names += [f"{f}={lv}" for lv in self.levels[f]]
        self.feature_names += CONT_FEATURES
        if self.include_biopsy:
            self.feature_names += ["biopsed_flag"]
        self.dim = len(self.feature_names)
        return self

    def transform_row(self, row):
        vec = []
        for f in CAT_FEATURES:
            val = str(row[f]) if pd.notna(row[f]) else self.mode[f]
            vec += [1.0 if val == lv else 0.0 for lv in self.levels[f]]
        for f in CONT_FEATURES:
            raw = row[f]
            vec.append(0.0 if pd.isna(raw) else (float(raw) - self.cmean[f]) / self.cstd[f])
        if self.include_biopsy:
            b = row.get("biopsed", np.nan)
            vec.append(1.0 if (pd.notna(b) and bool(b)) else 0.0)
        return np.asarray(vec, dtype=np.float32)


def compute_class_weights(train_idx):
    """Inverse-frequency class weights (mean-normalized) from train labels."""
    y = df.iloc[train_idx]["label"].values
    c = np.clip(np.bincount(y, minlength=CONFIG["num_classes"]), 1, None).astype(float)
    w = c.sum() / (CONFIG["num_classes"] * c); w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)


# ----- quick fit on the whole set to fix the GLOBAL vocabulary + record artifacts -----
_pp_global = MetaPreprocessor(include_biopsy=False).fit(df)
META_DIM = _pp_global.dim
artifacts = {"feature_names": _pp_global.feature_names, "meta_dim": META_DIM,
             "cat_features": CAT_FEATURES, "cont_features": CONT_FEATURES,
             "class_names": CLASS_NAMES, "class_to_idx": CLASS_TO_IDX}
with open(os.path.join(PATHS["metadata_analysis"], "preprocessing_artifacts.json"), "w") as f:
    json.dump(artifacts, f, indent=2)

print("Metadata feature dim (no biopsy):", META_DIM)
print("Example features:", _pp_global.feature_names[:6], "...", _pp_global.feature_names[-3:])
print("Saved preprocessing_artifacts.json")
print("CELL 4 complete.")

In [ ]:
# =====================================================================
# CELL 5 — SPLITTING


from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

labels = df["label"].values
groups = df["_patient_id"].values
all_idx = np.arange(len(df))

# ---------- Experiment 1: PATIENT-LEVEL split ----------
sgkf = StratifiedGroupKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=CONFIG["seed"])
folds = []                                  # patient-grouped folds (used for A0/A1/A2)
for fid, (trv, te) in enumerate(sgkf.split(all_idx, labels, groups)):
    # carve a patient-grouped validation set out of train
    inner = StratifiedGroupKFold(n_splits=int(round(1 / CONFIG["val_fraction"])),
                                 shuffle=True, random_state=CONFIG["seed"] + fid)
    tr_sub, va_sub = next(inner.split(trv, labels[trv], groups[trv]))
    folds.append({"fold": fid, "train": trv[tr_sub], "val": trv[va_sub], "test": te})

# integrity: no patient in two splits
for f in folds:
    s = {k: set(groups[f[k]]) for k in ["train", "val", "test"]}
    assert not (s["train"] & s["test"]) and not (s["val"] & s["test"]) and not (s["train"] & s["val"]), \
        f"patient leakage in fold {f['fold']}"
print("Patient-level split OK — zero patient overlap across train/val/test.")

# ---------- Experiment 2: RANDOM image-level split (intentionally leaky) ----------
skf = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=CONFIG["seed"])
folds_random = []
for fid, (trv, te) in enumerate(skf.split(all_idx, labels)):
    rng = np.random.RandomState(CONFIG["seed"] + fid)
    perm = rng.permutation(trv)
    n_val = int(CONFIG["val_fraction"] * len(perm))
    folds_random.append({"fold": fid, "train": perm[n_val:], "val": perm[:n_val], "test": te})
print("Random image-level split built (NOTE: patients may span splits — that's the point).")

# ---------- save split CSVs (reproducibility) ----------
def save_splits(fold_list, name):
    rows = []
    for f in fold_list:
        for split in ["train", "val", "test"]:
            for i in f[split]:
                rows.append({"fold": f["fold"], "split": split, "row_index": int(i),
                             "patient_id": df.iloc[i]["_patient_id"],
                             "label": int(df.iloc[i]["label"])})
    out = pd.DataFrame(rows)
    p = os.path.join(PATHS["fold_results"], f"splits_{name}.csv")
    out.to_csv(p, index=False); print(f"  saved {os.path.basename(p)} ({len(out)} rows)")
save_splits(folds, "patient_level")
save_splits(folds_random, "random_level")

# ---------- print distributions ----------
def dist(idx):
    sub = df.iloc[idx]
    pc = sub["diagnostic"].value_counts().reindex(CLASS_NAMES, fill_value=0)
    return len(sub), sub["_patient_id"].nunique(), pc

print("\n===== PATIENT-LEVEL split: per-fold TEST distribution =====")
hdr = f"{'fold':>4} {'imgs':>5} {'patients':>9}  " + " ".join(f"{c:>4}" for c in CLASS_NAMES)
print(hdr)
for f in folds:
    n, p, pc = dist(f["test"])
    print(f"{f['fold']:>4} {n:>5} {p:>9}  " + " ".join(f"{int(pc[c]):>4}" for c in CLASS_NAMES))

print("\nMelanoma (MEL) per TEST fold (n=52 total; watch for <8):")
for f in folds:
    mel = int((df.iloc[f["test"]]["diagnostic"] == "MEL").sum())
    print(f"  fold {f['fold']}: {mel}" + ("  <-- low" if mel < 8 else ""))

# sanity: show patient leakage rate in the RANDOM split (the thing we'll expose)
overlap_random = np.mean([
    len(set(groups[f["train"]]) & set(groups[f["test"]])) > 0 for f in folds_random])
print(f"\nRandom-split folds where a patient appears in BOTH train & test: "
      f"{overlap_random*100:.0f}% (this is the leakage we will demonstrate).")
print("CELL 5 complete.")

In [ ]:
# =====================================================================
# CELL 6 — IMAGE MODEL (A0)


class ImageModel(nn.Module):
    def __init__(self, num_classes=6, pretrained=True, freeze_early=True):
        super().__init__()
        # num_classes=0 + global_pool='' -> (B,1280,7,7) feature map (post conv_head)
        self.backbone = timm.create_model("efficientnet_b0", pretrained=pretrained,
                                           num_classes=0, global_pool="")
        self.feat_ch = self.backbone.num_features          # 1280
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(self.feat_ch, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes))
        if freeze_early:
            self._freeze_early()

    def _freeze_early(self):
        for p in self.backbone.conv_stem.parameters(): p.requires_grad = False
        for p in self.backbone.bn1.parameters():       p.requires_grad = False
        for i, blk in enumerate(self.backbone.blocks):
            for p in blk.parameters(): p.requires_grad = (i >= 4)   # train blocks 4,5,6

    def forward(self, image, meta=None):
        A = self.backbone(image)                           # (B,1280,7,7)
        feat = self.gap(A).flatten(1)                      # (B,1280)
        logits = self.classifier(feat)
        return {"logits": logits, "A": A}


_m = ImageModel(CONFIG["num_classes"], pretrained=False)
tot = sum(p.numel() for p in _m.parameters())
trn = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"A0 ImageModel  | total {tot/1e6:.2f}M | trainable {trn/1e6:.2f}M")
del _m; gc.collect()
print("CELL 6 complete.")

In [ ]:
# =====================================================================
# CELL 7 — METADATA MODEL (A1)

class MetadataModel(nn.Module):
    def __init__(self, meta_dim, num_classes=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(meta_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.3),
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, image=None, meta=None):
        h = self.net(meta)
        logits = self.classifier(h)
        return {"logits": logits, "A": None}


_m = MetadataModel(META_DIM, CONFIG["num_classes"])
tot = sum(p.numel() for p in _m.parameters())
print(f"A1 MetadataModel | meta_dim {META_DIM} | total {tot/1e3:.1f}K params")
del _m; gc.collect()
print("CELL 7 complete.")

In [ ]:
# =====================================================================
# CELL 8 — MULTIMODAL MODEL (A2)


class MultiModalModel(nn.Module):
    def __init__(self, meta_dim, num_classes=6, emb=256, pretrained=True, freeze_early=True):
        super().__init__()
        # image branch
        self.backbone = timm.create_model("efficientnet_b0", pretrained=pretrained,
                                           num_classes=0, global_pool="")
        self.feat_ch = self.backbone.num_features
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.img_proj = nn.Sequential(nn.Linear(self.feat_ch, emb), nn.ReLU())
        # metadata branch
        self.meta_mlp = nn.Sequential(
            nn.Linear(meta_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, emb), nn.ReLU())
        # fusion head over concatenated 2*emb
        self.classifier = nn.Sequential(
            nn.Linear(2 * emb, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes))
        if freeze_early:
            for p in self.backbone.conv_stem.parameters(): p.requires_grad = False
            for p in self.backbone.bn1.parameters():       p.requires_grad = False
            for i, blk in enumerate(self.backbone.blocks):
                for p in blk.parameters(): p.requires_grad = (i >= 4)

    def forward(self, image, meta):
        A = self.backbone(image)                           # (B,1280,7,7)
        g = self.img_proj(self.gap(A).flatten(1))          # (B,256)
        m = self.meta_mlp(meta)                            # (B,256)
        fused = torch.cat([g, m], dim=1)                   # (B,512)
        logits = self.classifier(fused)
        return {"logits": logits, "A": A, "img_emb": g, "meta_emb": m}


# tensor-dim + param print on a dummy batch
_m = MultiModalModel(META_DIM, CONFIG["num_classes"], pretrained=False).eval()
_img = torch.randn(4, 3, CONFIG["image_size"], CONFIG["image_size"])
_meta = torch.randn(4, META_DIM)
with torch.no_grad():
    _o = _m(_img, _meta)
print("A2 MultiModalModel tensor dims:")
print(f"  image in : {tuple(_img.shape)}")
print(f"  A (CAM)  : {tuple(_o['A'].shape)}")
print(f"  img_emb  : {tuple(_o['img_emb'].shape)}")
print(f"  meta_emb : {tuple(_o['meta_emb'].shape)}")
print(f"  fused    : {tuple(torch.cat([_o['img_emb'], _o['meta_emb']],1).shape)}")
print(f"  logits   : {tuple(_o['logits'].shape)}")
tot = sum(p.numel() for p in _m.parameters())
trn = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"  params   : total {tot/1e6:.2f}M | trainable {trn/1e6:.2f}M")
del _m, _o; gc.collect()
print("CELL 8 complete.")

In [ ]:
# =====================================================================
# CELL 9 — TRAINING COMPONENTS


def build_model(name, meta_dim=META_DIM, pretrained=True):
    """name in {'A0','A1','A2'} -> model on CONFIG['device']."""
    if name == "A0":   model = ImageModel(CONFIG["num_classes"], pretrained=pretrained)
    elif name == "A1": model = MetadataModel(meta_dim, CONFIG["num_classes"])
    elif name == "A2": model = MultiModalModel(meta_dim, CONFIG["num_classes"], pretrained=pretrained)
    else: raise ValueError(name)
    return model.to(CONFIG["device"])


def make_criterion(train_idx):
    w = compute_class_weights(train_idx).to(CONFIG["device"])
    return nn.CrossEntropyLoss(weight=w)


def build_optimizer(model):
    """Discriminative LRs: low for the pretrained backbone, high for new heads."""
    bb = [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad]
    hd = [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]
    groups = []
    if bb: groups.append({"params": bb, "lr": CONFIG["lr_backbone"]})
    groups.append({"params": hd, "lr": CONFIG["lr_head"]})
    return torch.optim.AdamW(groups, weight_decay=CONFIG["weight_decay"])


def build_scheduler(optimizer, n_steps):
    """Linear warmup -> cosine decay to ~1% of peak."""
    warmup = max(1, int(CONFIG["warmup_epochs"] * n_steps / CONFIG["epochs"]))
    def fn(step):
        if step < warmup: return step / warmup
        prog = (step - warmup) / max(1, n_steps - warmup)
        return 0.01 + 0.99 * 0.5 * (1 + np.cos(np.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


def save_checkpoint(model, meta, filename):
    """Save weights + reconstruction metadata so we can reload without retraining."""
    path = os.path.join(PATHS["models"], filename)
    torch.save({"state_dict": model.state_dict(), **meta}, path)
    return path


def load_checkpoint(filename, pretrained=False):
    """Rebuild the right architecture from saved meta and load weights."""
    path = os.path.join(PATHS["models"], filename)
    ck = torch.load(path, map_location=CONFIG["device"])
    model = build_model(ck["name"], ck.get("meta_dim", META_DIM), pretrained=pretrained)
    model.load_state_dict(ck["state_dict"]); model.eval()
    return model, ck


print("Training components ready: build_model / make_criterion / build_optimizer / "
      "build_scheduler / save_checkpoint / load_checkpoint")
print("CELL 9 complete.")

In [ ]:
# =====================================================================
# CELL 10 — TRAINING FUNCTIONS


from sklearn.metrics import f1_score as _f1


def _move(batch):
    return (batch["image"].to(CONFIG["device"], non_blocking=True),
            batch["meta"].to(CONFIG["device"], non_blocking=True),
            batch["label"].to(CONFIG["device"], non_blocking=True))


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, desc=""):
    model.train(); total, n = 0.0, 0
    for batch in tqdm(loader, desc=desc, leave=False):
        img, meta, y = _move(batch)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=CONFIG["use_amp"]):
            out = model(img, meta)
            loss = criterion(out["logits"], y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); scheduler.step()
        total += loss.item(); n += 1
    return total / max(1, n)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval(); total, n = 0.0, 0; ys, ps = [], []
    for batch in loader:
        img, meta, y = _move(batch)
        with autocast(enabled=CONFIG["use_amp"]):
            out = model(img, meta)
            loss = criterion(out["logits"], y)
        total += loss.item(); n += 1
        ps.append(out["logits"].argmax(1).cpu()); ys.append(y.cpu())
    y = torch.cat(ys).numpy(); p = torch.cat(ps).numpy()
    return total / max(1, n), _f1(y, p, average="macro")


def fit_model(name, loaders, train_idx, tag, fold_id, pretrained=True):
    """Train one model on one fold. Saves best_{tag}_fold{fold}.pt and a log CSV.
    Returns (best_checkpoint_filename, history list)."""
    model = build_model(name, META_DIM, pretrained=pretrained)
    criterion = make_criterion(train_idx)
    optimizer = build_optimizer(model)
    steps = len(loaders["train"])
    scheduler = build_scheduler(optimizer, CONFIG["epochs"] * steps)
    scaler = GradScaler(enabled=CONFIG["use_amp"])

    best_f1, best_file, wait, history = -1.0, None, 0, []
    ckpt_name = f"best_{tag}_fold{fold_id}.pt"

    for ep in range(CONFIG["epochs"]):
        tr_loss = train_one_epoch(model, loaders["train"], criterion, optimizer,
                                  scheduler, scaler, desc=f"{tag} f{fold_id} ep{ep+1}")
        va_loss, va_f1 = validate(model, loaders["val"], criterion)
        history.append({"epoch": ep + 1, "train_loss": tr_loss,
                        "val_loss": va_loss, "val_macroF1": va_f1})
        msg = (f"  [{tag} f{fold_id}] ep{ep+1:02d} | train_loss {tr_loss:.4f} | "
               f"val_loss {va_loss:.4f} | val_mF1 {va_f1:.4f}")

        if va_f1 > best_f1 + 1e-3:
            best_f1, wait = va_f1, 0
            best_file = save_checkpoint(model, {"name": name, "meta_dim": META_DIM,
                                                "tag": tag, "fold": fold_id,
                                                "val_macroF1": best_f1, "epoch": ep + 1},
                                        ckpt_name)
            print(msg + "  * best saved")
        else:
            wait += 1
            print(msg)
            if wait >= CONFIG["patience"]:
                print(f"  early stop at ep{ep+1} (best val_mF1 {best_f1:.4f})")
                break

    # save per-epoch training log (reproducible training curves without re-training)
    log_path = os.path.join(PATHS["logs"], f"trainlog_{tag}_fold{fold_id}.csv")
    pd.DataFrame(history).to_csv(log_path, index=False)
    print(f"  log -> {os.path.basename(log_path)} | model -> {ckpt_name}")
    del model, optimizer, scaler; gc.collect(); torch.cuda.empty_cache()
    return ckpt_name, history


print("Training functions ready: train_one_epoch / validate / fit_model")
print("CELL 10 complete.")

In [ ]:
# =====================================================================
# CELL 11 — DATALOADER FACTORY + TRAIN A0 (IMAGE ONLY)


IMAGENET_MEAN = [0.485, 0.456, 0.406]; IMAGENET_STD = [0.229, 0.224, 0.225]

def img_tf(train):
    if train:
        return T.Compose([T.Resize((CONFIG["image_size"], CONFIG["image_size"])),
                          T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                          T.RandomRotation(15), T.ColorJitter(0.1, 0.1),
                          T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    return T.Compose([T.Resize((CONFIG["image_size"], CONFIG["image_size"])),
                      T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


class PadDataset(Dataset):
    def __init__(self, frame, pp, train):
        self.frame = frame.reset_index(drop=True); self.pp = pp; self.tf = img_tf(train)
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = self.tf(Image.open(row["image_path"]).convert("RGB"))
        meta = torch.from_numpy(self.pp.transform_row(row))
        return {"image": img, "meta": meta,
                "label": torch.tensor(int(row["label"]), dtype=torch.long)}



GLOBAL_LEVELS = {f: list(_pp_global.levels[f]) for f in CAT_FEATURES}
assert _pp_global.dim == META_DIM, "META_DIM mismatch — check Cell 4"
print("GLOBAL_LEVELS built. META_DIM:", META_DIM)


def make_loaders(fold, include_biopsy=False):
    """Build train/val/test loaders for a fold. Vocabulary is FIXED globally;
    imputation stats (mode/mean/std) are fit on TRAIN rows only -> no leakage."""
    pp = MetaPreprocessor(include_biopsy=include_biopsy,
                          levels=GLOBAL_LEVELS).fit(df.iloc[fold["train"]])
    common = dict(num_workers=CONFIG["num_workers"], pin_memory=True)
    loaders = {
        "train": DataLoader(PadDataset(df.iloc[fold["train"]], pp, True),
                            batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True, **common),
        "val":   DataLoader(PadDataset(df.iloc[fold["val"]], pp, False),
                            batch_size=CONFIG["batch_size"], shuffle=False, **common),
        "test":  DataLoader(PadDataset(df.iloc[fold["test"]], pp, False),
                            batch_size=CONFIG["batch_size"], shuffle=False, **common)}
    return loaders, pp


def run_experiment(tag, model_name, fold_list, include_biopsy=False):
    ckpts = []
    for fold in fold_list:
        ckpt_name = f"best_{tag}_fold{fold['fold']}.pt"
        if os.path.exists(os.path.join(PATHS["models"], ckpt_name)):
            print(f"[skip] {ckpt_name} exists."); ckpts.append(ckpt_name); continue
        print(f"\n===== {tag} | fold {fold['fold']} =====")
        loaders, _ = make_loaders(fold, include_biopsy=include_biopsy)
        cn, _ = fit_model(model_name, loaders, fold["train"], tag, fold["fold"])
        ckpts.append(cn)
        gc.collect(); torch.cuda.empty_cache()
    return ckpts



for fold in folds:
    _, _pp = make_loaders(fold, include_biopsy=False)
    assert _pp.dim == META_DIM, f"fold {fold['fold']}: {_pp.dim} != {META_DIM}"
    print(f"fold {fold['fold']}: meta dim = {_pp.dim}  OK")

print("########## TRAINING A0 (IMAGE ONLY) ##########")
t0 = time.time()
A0_ckpts = run_experiment("A0", "A0", folds, include_biopsy=False)
print(f"\nA0 done in {(time.time()-t0)/60:.1f} min. Checkpoints: {A0_ckpts}")
print("CELL 11 complete.")

In [ ]:
# =====================================================================
# CELL 12 — TRAIN A1 (METADATA ONLY)


print("########## TRAINING A1 (METADATA ONLY) ##########")
t0 = time.time()
A1_ckpts = run_experiment("A1", "A1", folds, include_biopsy=False)
print(f"\nA1 done in {(time.time()-t0)/60:.1f} min. Checkpoints: {A1_ckpts}")
print("CELL 12 complete.")

In [ ]:
# =====================================================================
# CELL 13 — TRAIN A2 (IMAGE + METADATA FUSION)

print("########## TRAINING A2 (FUSION) ##########")
t0 = time.time()
A2_ckpts = run_experiment("A2", "A2", folds, include_biopsy=False)
print(f"\nA2 done in {(time.time()-t0)/60:.1f} min. Checkpoints: {A2_ckpts}")
print("CELL 13 complete.")

In [ ]:
# =====================================================================
# CELL 14 — EVALUATION


from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_recall_fscore_support, roc_auc_score,
                             confusion_matrix, classification_report)
from sklearn.preprocessing import label_binarize


@torch.no_grad()
def infer(model, loader):
    model.eval(); ys, prob = [], []
    for batch in loader:
        img = batch["image"].to(CONFIG["device"]); meta = batch["meta"].to(CONFIG["device"])
        with autocast(enabled=CONFIG["use_amp"]):
            out = model(img, meta)
        prob.append(out["logits"].softmax(1).float().cpu()); ys.append(batch["label"])
    return torch.cat(ys).numpy(), torch.cat(prob).numpy()


def full_metrics(y, prob):
    pred = prob.argmax(1)
    acc = accuracy_score(y, pred); bacc = balanced_accuracy_score(y, pred)
    mp, mr, mf1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    yb = label_binarize(y, classes=list(range(CONFIG["num_classes"])))
    pres = yb.sum(0) > 0
    try: auc = roc_auc_score(yb[:, pres], prob[:, pres], average="macro", multi_class="ovr")
    except ValueError: auc = float("nan")
    pc = precision_recall_fscore_support(y, pred, labels=list(range(CONFIG["num_classes"])),
                                         zero_division=0)
    return {"accuracy": acc, "balanced_accuracy": bacc, "macro_precision": mp,
            "macro_recall": mr, "macro_f1": mf1, "auroc": auc,
            "per_class_precision": pc[0].tolist(), "per_class_recall": pc[1].tolist(),
            "per_class_f1": pc[2].tolist(), "support": pc[3].tolist()}


def evaluate_experiment(tag, fold_list, ref_fold=0):
    """Evaluate every fold checkpoint of `tag`; save predictions + metrics."""
    all_metrics = []
    for fold in fold_list:
        ckpt_name = f"best_{tag}_fold{fold['fold']}.pt"
        if not os.path.exists(os.path.join(PATHS["models"], ckpt_name)):
            print(f"[warn] missing {ckpt_name}, skipping."); continue
        model, ck = load_checkpoint(ckpt_name)
        loaders, _ = make_loaders(fold, include_biopsy=False)
        y, prob = infer(model, loaders["test"]); del model; gc.collect(); torch.cuda.empty_cache()

        # save predictions (reproducibility: all later figures use these)
        pred_df = pd.DataFrame({"y_true": y, "y_pred": prob.argmax(1)})
        for c in range(CONFIG["num_classes"]):
            pred_df[f"prob_{CLASS_NAMES[c]}"] = prob[:, c]
        pred_path = os.path.join(PATHS["predictions"], f"pred_{tag}_fold{fold['fold']}.csv")
        pred_df.to_csv(pred_path, index=False)

        m = full_metrics(y, prob); m.update({"tag": tag, "fold": fold["fold"]})
        all_metrics.append(m)

        # save per-fold metrics json
        with open(os.path.join(PATHS["metrics"], f"metrics_{tag}_fold{fold['fold']}.json"), "w") as fjs:
            json.dump(m, fjs, indent=2)

        # print reference-fold confusion matrix + classification report
        if fold["fold"] == ref_fold:
            cm = confusion_matrix(y, prob.argmax(1), labels=list(range(CONFIG["num_classes"])))
            print(f"\n----- {tag} fold {ref_fold}: confusion matrix -----")
            print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_string())
            print(f"\n----- {tag} fold {ref_fold}: classification report -----")
            print(classification_report(y, prob.argmax(1), target_names=CLASS_NAMES, zero_division=0))
            np.save(os.path.join(PATHS["metrics"], f"cm_{tag}_fold{ref_fold}.npy"), cm)
    return pd.DataFrame(all_metrics)

print("########## EVALUATING A0 / A1 / A2 ##########")
metrics_A0 = evaluate_experiment("A0", folds)
metrics_A1 = evaluate_experiment("A1", folds)
metrics_A2 = evaluate_experiment("A2", folds)
ALL_METRICS = pd.concat([metrics_A0, metrics_A1, metrics_A2], ignore_index=True)
ALL_METRICS.to_csv(os.path.join(PATHS["metrics"], "all_fold_metrics.csv"), index=False)
print("\nSaved per-fold predictions (predictions/) and metrics (metrics/all_fold_metrics.csv).")
print("CELL 14 complete.")

In [ ]:
# =====================================================================
# CELL 15 — CROSS-FOLD SUMMARY


# reload from disk if needed (crash-safe)
if "ALL_METRICS" not in dir() or ALL_METRICS is None or len(ALL_METRICS) == 0:
    ALL_METRICS = pd.read_csv(os.path.join(PATHS["metrics"], "all_fold_metrics.csv"))

SCALAR_METRICS = ["accuracy", "balanced_accuracy", "macro_precision",
                  "macro_recall", "macro_f1", "auroc"]

# ---------- (1) fold-wise table ----------
foldwise = ALL_METRICS[["tag", "fold"] + SCALAR_METRICS].sort_values(["tag", "fold"]).round(4)
foldwise.to_csv(os.path.join(PATHS["tables"], "foldwise_metrics.csv"), index=False)
print("===== FOLD-WISE METRICS =====")
print(foldwise.to_string(index=False))

# ---------- (2) mean ± std per experiment ----------
rows = []
for tag in ["A0", "A1", "A2"]:
    sub = ALL_METRICS[ALL_METRICS.tag == tag]
    row = {"experiment": tag}
    for mname in SCALAR_METRICS:
        row[f"{mname}_mean"] = round(sub[mname].mean(), 4)
        row[f"{mname}_std"] = round(sub[mname].std(), 4)
    rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(PATHS["tables"], "mean_std_summary.csv"), index=False)

print("\n===== MEAN ± STD (5-fold) =====")
for _, r in summary.iterrows():
    print(f" {r['experiment']}: "
          f"Acc {r['accuracy_mean']:.3f}±{r['accuracy_std']:.3f} | "
          f"BAcc {r['balanced_accuracy_mean']:.3f}±{r['balanced_accuracy_std']:.3f} | "
          f"mF1 {r['macro_f1_mean']:.3f}±{r['macro_f1_std']:.3f} | "
          f"AUROC {r['auroc_mean']:.3f}±{r['auroc_std']:.3f}")

# ---------- (3) best model (by mean macro-F1) ----------
best_tag = summary.loc[summary["macro_f1_mean"].idxmax(), "experiment"]
best_f1 = summary["macro_f1_mean"].max()
print(f"\nBest experiment by mean macro-F1: {best_tag} ({best_f1:.4f})")

# paper-ready combined summary
paper_tbl = summary.copy()
paper_tbl.to_csv(os.path.join(PATHS["paper_outputs"], "table_main_results.csv"), index=False)
with open(os.path.join(PATHS["paper_outputs"], "best_model.txt"), "w") as f:
    f.write(f"Best experiment (mean macro-F1): {best_tag} = {best_f1:.4f}\n")
    f.write(summary.to_string(index=False))

print("\nSaved: tables/foldwise_metrics.csv, tables/mean_std_summary.csv,")
print("       paper_outputs/table_main_results.csv, paper_outputs/best_model.txt")
print("CELL 15 complete.")

In [ ]:
# =====================================================================
# CELL 16 — GRAD-CAM


IMAGENET_MEAN_T = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD_T = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
def denorm(t): return (t.cpu() * IMAGENET_STD_T + IMAGENET_MEAN_T).clamp(0, 1).permute(1, 2, 0).numpy()


class GradCAM:
    def __init__(self, model):
        self.model = model.eval(); self.act = self.grad = None
        self.h = model.backbone.register_forward_hook(self._hook)
    def _hook(self, m, i, o):
        self.act = o; o.register_hook(lambda g: setattr(self, "grad", g))
    def __call__(self, image, meta):
        self.model.zero_grad()
        with torch.enable_grad():
            out = self.model(image, meta); cls = out["logits"].argmax(1)
            out["logits"].gather(1, cls.view(-1, 1)).sum().backward()
        w = self.grad.mean((2, 3), keepdim=True)
        cam = F.relu((w * self.act).sum(1))
        cam = cam / (cam.amax((1, 2), keepdim=True) + 1e-8)
        cam = F.interpolate(cam.unsqueeze(1), size=image.shape[-2:], mode="bilinear",
                            align_corners=False).squeeze(1)
        return cam.detach().cpu().numpy(), cls.cpu().numpy()
    def close(self): self.h.remove()


def border_attention_fraction(cam, border=0.18):
    """Proxy for artifact attention: fraction of CAM mass in the outer border ring."""
    h, w = cam.shape; bh, bw = int(h * border), int(w * border)
    mask = np.ones_like(cam); mask[bh:h-bh, bw:w-bw] = 0
    return float((cam * mask).sum() / (cam.sum() + 1e-8))


import matplotlib.pyplot as plt
matplotlib_ok = True

model_a2, _ = load_checkpoint("best_A2_fold0.pt")
loaders0, _pp0 = make_loaders(folds[0], include_biopsy=False)
ds = loaders0["test"].dataset

# get fold-0 predictions to pick correct + failure cases
pred0 = pd.read_csv(os.path.join(PATHS["predictions"], "pred_A2_fold0.csv"))
wrong = np.where(pred0["y_true"].values != pred0["y_pred"].values)[0]
right = np.where(pred0["y_true"].values == pred0["y_pred"].values)[0]
pick = list(right[:4]) + (list(wrong[:1]) if len(wrong) else [])

cam_engine = GradCAM(model_a2)
border_fracs = []
fig, axes = plt.subplots(len(pick), 3, figsize=(6.5, 2.0 * len(pick)), dpi=300)
for r, i in enumerate(pick):
    s = ds[i]
    cam, cls = cam_engine(s["image"].unsqueeze(0).to(CONFIG["device"]),
                          s["meta"].unsqueeze(0).to(CONFIG["device"]))
    base = denorm(s["image"]); bf = border_attention_fraction(cam[0]); border_fracs.append(bf)
    ok = int(cls[0]) == s["label"]
    axes[r, 0].imshow(base); axes[r, 0].set_title("image", fontsize=7)
    axes[r, 1].imshow(cam[0], cmap="jet"); axes[r, 1].set_title("Grad-CAM", fontsize=7)
    axes[r, 2].imshow(base); axes[r, 2].imshow(cam[0], cmap="jet", alpha=0.45)
    axes[r, 2].set_title(f"true={CLASS_NAMES[s['label']]} pred={CLASS_NAMES[int(cls[0])]}"
                         f"{'' if ok else '  ✗'}  border={bf:.2f}",
                         fontsize=7, color="black" if ok else "red")
    for a in axes[r]: a.axis("off")
fig.tight_layout()
for ext in ["png", "pdf"]:
    fig.savefig(os.path.join(PATHS["gradcam"], f"gradcam_examples.{ext}"), bbox_inches="tight")
plt.close(); cam_engine.close()

# save the artifact-attention numbers
pd.DataFrame({"example": list(range(len(pick))), "border_attention_fraction": border_fracs}
             ).to_csv(os.path.join(PATHS["gradcam"], "artifact_attention.csv"), index=False)
print(f"Saved gradcam_examples.(png/pdf) | mean border-attention fraction: "
      f"{np.mean(border_fracs):.3f} (higher -> more attention on image edges/artifacts)")
del model_a2; gc.collect(); torch.cuda.empty_cache()
print("CELL 16 complete.")

In [ ]:
# =====================================================================
# CELL 17 — METADATA IMPORTANCE (permutation; SHAP-free, no NumPy conflicts)

model_a2, _ = load_checkpoint("best_A2_fold0.pt")
loaders0, pp0 = make_loaders(folds[0], include_biopsy=False)
test_frame = df.iloc[folds[0]["test"]].reset_index(drop=True)

# encode metadata matrix once + preload fixed images
M = np.stack([pp0.transform_row(r) for _, r in test_frame.iterrows()]).astype(np.float32)
imgs = torch.stack([img_tf(False)(Image.open(r["image_path"]).convert("RGB"))
                    for _, r in test_frame.iterrows()])

# map one-hot columns -> parent feature names (for aggregation)
col_to_feature = []
for f in CAT_FEATURES: col_to_feature += [f] * len(pp0.levels[f])
col_to_feature += CONT_FEATURES

@torch.no_grad()
def predicted_class_prob(matrix):
    out_p = []
    for s in range(0, len(matrix), CONFIG["batch_size"]):
        sl = slice(s, s + CONFIG["batch_size"])
        mt = torch.tensor(matrix[sl], device=CONFIG["device"])
        with autocast(enabled=CONFIG["use_amp"]):
            o = model_a2(imgs[sl].to(CONFIG["device"]), mt)
        out_p.append(o["logits"].softmax(1).float().cpu())
    return torch.cat(out_p).numpy()

base_p = predicted_class_prob(M); tgt = base_p.argmax(1)
base_score = base_p[np.arange(len(tgt)), tgt].mean()

rng = np.random.RandomState(CONFIG["seed"]); N_REPEAT = 3
col_imp = np.zeros(M.shape[1])
for c in tqdm(range(M.shape[1]), desc="perm-importance"):
    drops = []
    for _ in range(N_REPEAT):
        Mp = M.copy(); Mp[:, c] = Mp[rng.permutation(len(Mp)), c]
        p = predicted_class_prob(Mp); drops.append(base_score - p[np.arange(len(tgt)), tgt].mean())
    col_imp[c] = max(0.0, float(np.mean(drops)))

# aggregate one-hot columns -> per-feature importance
feat_imp = {}
for name, v in zip(col_to_feature, col_imp): feat_imp[name] = feat_imp.get(name, 0.0) + v
imp_df = (pd.DataFrame({"feature": list(feat_imp.keys()), "importance": list(feat_imp.values())})
          .sort_values("importance", ascending=False).reset_index(drop=True))
imp_df.to_csv(os.path.join(PATHS["metadata_analysis"], "metadata_importance.csv"), index=False)

print("===== METADATA FEATURE IMPORTANCE (permutation) =====")
print(imp_df.to_string(index=False))
del model_a2; gc.collect(); torch.cuda.empty_cache()
print("CELL 17 complete.")

In [ ]:
# =====================================================================
# CELL 18 — FEATURE ABLATION


from sklearn.metrics import f1_score

model_a2, _ = load_checkpoint("best_A2_fold0.pt")
loaders0, pp0 = make_loaders(folds[0], include_biopsy=False)
test_frame = df.iloc[folds[0]["test"]].reset_index(drop=True)
y_true = test_frame["label"].values

M = np.stack([pp0.transform_row(r) for _, r in test_frame.iterrows()]).astype(np.float32)
imgs = torch.stack([img_tf(False)(Image.open(r["image_path"]).convert("RGB"))
                    for _, r in test_frame.iterrows()])

# build column index ranges per feature
feature_cols = {}
c = 0
for f in CAT_FEATURES:
    w = len(pp0.levels[f]); feature_cols[f] = list(range(c, c + w)); c += w
for f in CONT_FEATURES:
    feature_cols[f] = [c]; c += 1

@torch.no_grad()
def macro_f1_for(matrix):
    preds = []
    for s in range(0, len(matrix), CONFIG["batch_size"]):
        sl = slice(s, s + CONFIG["batch_size"])
        mt = torch.tensor(matrix[sl], device=CONFIG["device"])
        with autocast(enabled=CONFIG["use_amp"]):
            o = model_a2(imgs[sl].to(CONFIG["device"]), mt)
        preds.append(o["logits"].argmax(1).cpu())
    return f1_score(y_true, torch.cat(preds).numpy(), average="macro")

base_f1 = macro_f1_for(M)
rows = []
for f in tqdm(list(feature_cols.keys()), desc="feature-ablation"):
    Mab = M.copy(); Mab[:, feature_cols[f]] = 0.0          # remove this feature
    f1_ab = macro_f1_for(Mab)
    rows.append({"feature": f, "macro_f1_without": round(f1_ab, 4),
                 "drop_from_full": round(base_f1 - f1_ab, 4)})

abl_df = pd.DataFrame(rows).sort_values("drop_from_full", ascending=False).reset_index(drop=True)
abl_df.to_csv(os.path.join(PATHS["metadata_analysis"], "feature_ablation.csv"), index=False)
print(f"Full-metadata macro-F1 (A2, fold0): {base_f1:.4f}\n")
print("===== FEATURE ABLATION (drop = full - without) =====")
print(abl_df.to_string(index=False))
del model_a2; gc.collect(); torch.cuda.empty_cache()
print("CELL 18 complete.")

In [ ]:
# =====================================================================
# CELL 19 — METADATA DEGRADATION


model_a2, _ = load_checkpoint("best_A2_fold0.pt")
loaders0, pp0 = make_loaders(folds[0], include_biopsy=False)
test_frame = df.iloc[folds[0]["test"]].reset_index(drop=True)
y_true = test_frame["label"].values

M = np.stack([pp0.transform_row(r) for _, r in test_frame.iterrows()]).astype(np.float32)
imgs = torch.stack([img_tf(False)(Image.open(r["image_path"]).convert("RGB"))
                    for _, r in test_frame.iterrows()])
n_cols = M.shape[1]

@torch.no_grad()
def eval_matrix(matrix):
    preds = []
    for s in range(0, len(matrix), CONFIG["batch_size"]):
        sl = slice(s, s + CONFIG["batch_size"])
        mt = torch.tensor(matrix[sl], device=CONFIG["device"])
        with autocast(enabled=CONFIG["use_amp"]):
            o = model_a2(imgs[sl].to(CONFIG["device"]), mt)
        preds.append(o["logits"].argmax(1).cpu())
    p = torch.cat(preds).numpy()
    return (f1_score(y_true, p, average="macro"), balanced_accuracy_score(y_true, p))

REMOVE = [0.0, 0.25, 0.50, 0.75, 1.0]; SEEDS = [0, 1, 2]
rows = []
for frac in REMOVE:
    mf1s, baccs = [], []
    for sd in SEEDS:
        rng = np.random.RandomState(sd)
        Md = M.copy()
        if frac > 0:
            for i in range(len(Md)):                       # per-sample random removal
                k = int(round(frac * n_cols))
                cols = rng.choice(n_cols, k, replace=False); Md[i, cols] = 0.0
        f1v, bacc = eval_matrix(Md); mf1s.append(f1v); baccs.append(bacc)
    rows.append({"metadata_removed": frac,
                 "macro_f1_mean": round(np.mean(mf1s), 4), "macro_f1_std": round(np.std(mf1s), 4),
                 "balanced_acc_mean": round(np.mean(baccs), 4), "balanced_acc_std": round(np.std(baccs), 4)})
    print(f"  removed {int(frac*100):3d}% -> macro-F1 {np.mean(mf1s):.4f} | bAcc {np.mean(baccs):.4f}")

deg_df = pd.DataFrame(rows)
deg_df.to_csv(os.path.join(PATHS["metadata_analysis"], "metadata_degradation.csv"), index=False)
print("\nSaved metadata_degradation.csv")
del model_a2; gc.collect(); torch.cuda.empty_cache()
print("CELL 19 complete.")

In [ ]:
# =====================================================================
# CELL 20 — METADATA ANALYSIS PLOTS + CSV CONSOLIDATION


imp_df = pd.read_csv(os.path.join(PATHS["metadata_analysis"], "metadata_importance.csv"))
abl_df = pd.read_csv(os.path.join(PATHS["metadata_analysis"], "feature_ablation.csv"))
deg_df = pd.read_csv(os.path.join(PATHS["metadata_analysis"], "metadata_degradation.csv"))

def save_fig(fig, name):
    for ext in ["png", "pdf"]:
        fig.savefig(os.path.join(PATHS["figures"], f"{name}.{ext}"), bbox_inches="tight")
    plt.close(fig); print(f"  saved {name}.png/.pdf")

# ---- importance ----
top = imp_df.head(15)[::-1]
fig, ax = plt.subplots(figsize=(5.2, 5), dpi=300)
ax.barh(top["feature"], top["importance"], color="#2c7fb8")
ax.set_xlabel("permutation importance (Δ pred prob)")
ax.set_title("Metadata feature importance (A2, fold 0)"); ax.tick_params(axis="y", labelsize=7)
fig.tight_layout(); save_fig(fig, "fig_metadata_importance")

# ---- feature ablation ----
topa = abl_df.head(15)[::-1]
fig, ax = plt.subplots(figsize=(5.2, 5), dpi=300)
ax.barh(topa["feature"], topa["drop_from_full"], color="#e6550d")
ax.set_xlabel("macro-F1 drop when feature removed")
ax.set_title("Per-feature ablation (A2, fold 0)"); ax.tick_params(axis="y", labelsize=7)
fig.tight_layout(); save_fig(fig, "fig_feature_ablation")

# ---- degradation curve ----
fig, ax = plt.subplots(figsize=(5.4, 3.6), dpi=300)
x = deg_df["metadata_removed"] * 100
ax.errorbar(x, deg_df["macro_f1_mean"], yerr=deg_df["macro_f1_std"],
            marker="o", capsize=3, color="#2c7fb8", label="Macro-F1")
ax.errorbar(x, deg_df["balanced_acc_mean"], yerr=deg_df["balanced_acc_std"],
            marker="s", capsize=3, color="#31a354", label="Balanced Acc")
ax.set_xlabel("% metadata removed"); ax.set_ylabel("score")
ax.set_title("Performance vs. metadata degradation (A2)"); ax.legend(fontsize=8)
ax.invert_xaxis() if False else None
fig.tight_layout(); save_fig(fig, "fig_metadata_degradation")

# ---- consolidate paper copies ----
imp_df.to_csv(os.path.join(PATHS["paper_outputs"], "table_metadata_importance.csv"), index=False)
abl_df.to_csv(os.path.join(PATHS["paper_outputs"], "table_feature_ablation.csv"), index=False)
deg_df.to_csv(os.path.join(PATHS["paper_outputs"], "table_metadata_degradation.csv"), index=False)
print("\nSaved 3 figures (png+pdf) and 3 paper tables.")
print("CELL 20 complete.")

In [ ]:
# =====================================================================
# CELL 21 — LEAKAGE STUDY: PATIENT-LEVEL vs RANDOM SPLIT

groups = df["_patient_id"].values

# ---- train A2 under the random (leaky) split ----
print("########## TRAINING A2 under RANDOM split (leakage demo) ##########")
t0 = time.time()
for fold in folds_random:
    ckpt = f"best_A2rand_fold{fold['fold']}.pt"
    if os.path.exists(os.path.join(PATHS["models"], ckpt)):
        print(f"[skip] {ckpt} exists."); continue
    loaders, _ = make_loaders(fold, include_biopsy=False)
    fit_model("A2", loaders, fold["train"], "A2rand", fold["fold"])
    gc.collect(); torch.cuda.empty_cache()
print(f"Random-split A2 done in {(time.time()-t0)/60:.1f} min.")

# ---- evaluate random-split A2 ----
metrics_A2rand = evaluate_experiment("A2rand", folds_random)

# ---- build comparison (patient vs random), reload patient metrics if needed ----
if "ALL_METRICS" not in dir():
    ALL_METRICS = pd.read_csv(os.path.join(PATHS["metrics"], "all_fold_metrics.csv"))
pat = ALL_METRICS[ALL_METRICS.tag == "A2"]
rnd = metrics_A2rand

def msd(sub, m): return sub[m].mean(), sub[m].std()
rows = []
for mname in ["accuracy", "balanced_accuracy", "macro_f1", "auroc"]:
    pm, ps = msd(pat, mname); rm, rs = msd(rnd, mname)
    rows.append({"metric": mname,
                 "patient_split": f"{pm:.3f}±{ps:.3f}",
                 "random_split": f"{rm:.3f}±{rs:.3f}",
                 "inflation": round(rm - pm, 4)})
leak_df = pd.DataFrame(rows)
leak_df.to_csv(os.path.join(PATHS["tables"], "leakage_split_comparison.csv"), index=False)
leak_df.to_csv(os.path.join(PATHS["paper_outputs"], "table_leakage_split.csv"), index=False)

# also record the patient-overlap rate (the mechanism)
overlap_rate = float(np.mean([
    len(set(groups[f["train"]]) & set(groups[f["test"]])) > 0 for f in folds_random]))

print("\n===== LEAKAGE: PATIENT vs RANDOM SPLIT (A2) =====")
print(leak_df.to_string(index=False))
print(f"\nRandom-split folds with a patient in BOTH train & test: {overlap_rate*100:.0f}%")
print("Positive 'inflation' = how much the improper random split OVERESTIMATES performance.")
print("CELL 21 complete.")

In [ ]:
# =====================================================================
# CELL 22 — BIOPSY-FLAG LEAKAGE DEMONSTRATION (with vs without biopsed)


import time
print("########## TRAINING A2 WITH biopsy flag (leakage demo) ##########")
t0 = time.time()

def fit_model_biopsy(fold):
    pp = MetaPreprocessor(include_biopsy=True, levels=GLOBAL_LEVELS).fit(df.iloc[fold["train"]])
    bdim = pp.dim
    common = dict(num_workers=CONFIG["num_workers"], pin_memory=True)
    loaders = {
        "train": DataLoader(PadDataset(df.iloc[fold["train"]], pp, True),
                            batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True, **common),
        "val":   DataLoader(PadDataset(df.iloc[fold["val"]], pp, False),
                            batch_size=CONFIG["batch_size"], shuffle=False, **common),
        "test":  DataLoader(PadDataset(df.iloc[fold["test"]], pp, False),
                            batch_size=CONFIG["batch_size"], shuffle=False, **common)}
    model = MultiModalModel(bdim, CONFIG["num_classes"], pretrained=True).to(CONFIG["device"])
    criterion = make_criterion(fold["train"])
    optimizer = build_optimizer(model); steps = len(loaders["train"])
    scheduler = build_scheduler(optimizer, CONFIG["epochs"] * steps)
    scaler = GradScaler(enabled=CONFIG["use_amp"])
    best, wait, ckpt_name = -1, 0, f"best_A2biopsy_fold{fold['fold']}.pt"
    for ep in range(CONFIG["epochs"]):
        train_one_epoch(model, loaders["train"], criterion, optimizer, scheduler, scaler,
                        desc=f"A2biopsy f{fold['fold']} ep{ep+1}")
        _, vf1 = validate(model, loaders["val"], criterion)
        if vf1 > best + 1e-3:
            best, wait = vf1, 0
            torch.save({"state_dict": model.state_dict(), "name": "A2",
                        "meta_dim": bdim, "tag": "A2biopsy", "fold": fold["fold"],
                        "val_macroF1": best}, os.path.join(PATHS["models"], ckpt_name))
        else:
            wait += 1
            if wait >= CONFIG["patience"]: break
    ck = torch.load(os.path.join(PATHS["models"], ckpt_name), map_location=CONFIG["device"])
    model.load_state_dict(ck["state_dict"]); model.eval()
    y, prob = infer(model, loaders["test"])
    del model; gc.collect(); torch.cuda.empty_cache()
    return full_metrics(y, prob)


def eval_existing_biopsy(fold):
    """Re-evaluate an already-trained biopsy checkpoint (no retraining)."""
    ckpt_name = f"best_A2biopsy_fold{fold['fold']}.pt"
    ck = torch.load(os.path.join(PATHS["models"], ckpt_name), map_location=CONFIG["device"])
    model = MultiModalModel(ck["meta_dim"], CONFIG["num_classes"], pretrained=False).to(CONFIG["device"])
    model.load_state_dict(ck["state_dict"]); model.eval()
    pp = MetaPreprocessor(include_biopsy=True, levels=GLOBAL_LEVELS).fit(df.iloc[fold["train"]])
    loaders = {"test": DataLoader(PadDataset(df.iloc[fold["test"]], pp, False),
                                  batch_size=CONFIG["batch_size"], shuffle=False,
                                  num_workers=CONFIG["num_workers"], pin_memory=True)}
    y, prob = infer(model, loaders["test"]); del model; gc.collect(); torch.cuda.empty_cache()
    return full_metrics(y, prob)


biopsy_rows = []
for fold in folds:
    ckpt_name = f"best_A2biopsy_fold{fold['fold']}.pt"
    if os.path.exists(os.path.join(PATHS["models"], ckpt_name)):
        print(f"[skip] {ckpt_name} exists.")
        m = eval_existing_biopsy(fold)
    else:
        m = fit_model_biopsy(fold)
    m.update({"tag": "A2biopsy", "fold": fold["fold"]})
    biopsy_rows.append(m)

metrics_biopsy = pd.DataFrame(biopsy_rows)
print(f"Biopsy-flag A2 done in {(time.time()-t0)/60:.1f} min.")

# ---- comparison: clean A2 vs A2+biopsy ----
clean = ALL_METRICS[ALL_METRICS.tag == "A2"]
rows = []
for mname in ["accuracy", "balanced_accuracy", "macro_f1", "auroc"]:
    cm_, cs_ = clean[mname].mean(), clean[mname].std()
    bm_, bs_ = metrics_biopsy[mname].mean(), metrics_biopsy[mname].std()
    rows.append({"metric": mname,
                 "without_biopsy": f"{cm_:.3f}±{cs_:.3f}",
                 "with_biopsy": f"{bm_:.3f}±{bs_:.3f}",
                 "inflation": round(bm_ - cm_, 4)})
biopsy_df = pd.DataFrame(rows)
biopsy_df.to_csv(os.path.join(PATHS["paper_outputs"], "table_leakage_biopsy.csv"), index=False)
print("\n===== LEAKAGE: WITH vs WITHOUT BIOPSY FLAG (A2) =====")
print(biopsy_df.to_string(index=False))
print("Positive 'inflation' = how much including the biopsy flag OVERESTIMATES performance.")
print("CELL 22 complete.")

In [ ]:
# =====================================================================
# CELL 23 — CLASS-WISE MODALITY ANALYSIS

if "ALL_METRICS" not in dir():
    ALL_METRICS = pd.read_csv(os.path.join(PATHS["metrics"], "all_fold_metrics.csv"))

def _parse_list(v):
    """Robustly parse per_class_f1, whether it's already a list (in-memory)
    or a string (loaded from CSV), without using eval()."""
    if isinstance(v, list):
        return v
    if isinstance(v, str):
        return json.loads(v)               # CSV-saved lists are valid JSON
    raise ValueError(f"Unexpected per_class_f1 type: {type(v)} -> {v!r}")

def per_class_mean(tag):
    """Average per-class F1 across folds for a tag."""
    sub = ALL_METRICS[ALL_METRICS.tag == tag]
    if len(sub) == 0:
        raise ValueError(f"No rows found for tag={tag!r} in ALL_METRICS.")
    arrs = [_parse_list(v) for v in sub["per_class_f1"]]
    return np.mean(np.array(arrs, dtype=float), axis=0)

rows = []
f1_A0, f1_A1, f1_A2 = per_class_mean("A0"), per_class_mean("A1"), per_class_mean("A2")
for i, c in enumerate(CLASS_NAMES):
    rows.append({"class": c,
                 "image_only_f1": round(f1_A0[i], 3),
                 "metadata_only_f1": round(f1_A1[i], 3),
                 "fusion_f1": round(f1_A2[i], 3),
                 "fusion_gain_over_best_unimodal": round(f1_A2[i] - max(f1_A0[i], f1_A1[i]), 3)})
classwise = pd.DataFrame(rows)
classwise.to_csv(os.path.join(PATHS["paper_outputs"], "table_classwise_modality.csv"), index=False)

print("===== CLASS-WISE MODALITY F1 (mean over folds) =====")
print(classwise.to_string(index=False))
mel = classwise[classwise["class"] == "MEL"]
if len(mel):
    r = mel.iloc[0]
    print(f"\n>>> MELANOMA: image {r['image_only_f1']:.3f} | metadata {r['metadata_only_f1']:.3f} | "
          f"fusion {r['fusion_f1']:.3f}  (n=52 total — interpret with wide CI)")
print("CELL 23 complete.")

In [ ]:
# =====================================================================
# CELL 24 — PAPER FIGURES (png + pdf)


from sklearn.metrics import roc_curve, auc as _auc, confusion_matrix
from sklearn.preprocessing import label_binarize

def save_fig(fig, name):
    for ext in ["png", "pdf"]:
        fig.savefig(os.path.join(PATHS["figures"], f"{name}.{ext}"), bbox_inches="tight")
    plt.close(fig); print(f"  saved {name}")

# ---- ROC (A2, pooled over folds from saved predictions) ----
prob_all, y_all = [], []
for f in folds:
    p = pd.read_csv(os.path.join(PATHS["predictions"], f"pred_A2_fold{f['fold']}.csv"))
    y_all.append(p["y_true"].values)
    prob_all.append(p[[f"prob_{c}" for c in CLASS_NAMES]].values)
y_all = np.concatenate(y_all); prob_all = np.concatenate(prob_all)
yb = label_binarize(y_all, classes=list(range(CONFIG["num_classes"])))
fig, ax = plt.subplots(figsize=(4.6, 4.4), dpi=300)
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
for i, c in enumerate(CLASS_NAMES):
    if yb[:, i].sum() == 0: continue
    fpr, tpr, _ = roc_curve(yb[:, i], prob_all[:, i])
    ax.plot(fpr, tpr, lw=1.3, label=f"{c} (AUC={_auc(fpr,tpr):.2f})")
ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
ax.set_title("ROC — A2 (pooled folds)"); ax.legend(fontsize=7, loc="lower right")
fig.tight_layout(); save_fig(fig, "fig_roc")

# ---- aggregated confusion matrix (A2, pooled) ----
cm = confusion_matrix(y_all, prob_all.argmax(1), labels=list(range(CONFIG["num_classes"])))
cmn = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
fig, ax = plt.subplots(figsize=(4.8, 4.2), dpi=300)
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix — A2 (pooled)")
fig.colorbar(im, fraction=0.046, pad=0.04); fig.tight_layout(); save_fig(fig, "fig_confusion_aggregated")

# ---- training curves (A2 fold 0 from saved log) ----
log0 = pd.read_csv(os.path.join(PATHS["logs"], "trainlog_A2_fold0.csv"))
fig, ax1 = plt.subplots(figsize=(5.2, 3.4), dpi=300)
ax1.plot(log0["epoch"], log0["train_loss"], color="#e6550d", marker="o", ms=3, label="train loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train loss", color="#e6550d")
ax2 = ax1.twinx()
ax2.plot(log0["epoch"], log0["val_macroF1"], color="#2c7fb8", marker="s", ms=3, label="val macro-F1")
ax2.set_ylabel("val macro-F1", color="#2c7fb8")
ax1.set_title("Training curves — A2 fold 0"); fig.tight_layout(); save_fig(fig, "fig_training_curves")

# ---- modality comparison bar chart ----
summ = pd.read_csv(os.path.join(PATHS["tables"], "mean_std_summary.csv"))
fig, ax = plt.subplots(figsize=(6, 3.8), dpi=300)
x = np.arange(len(summ)); w = 0.38
ax.bar(x - w/2, summ["balanced_accuracy_mean"], w, yerr=summ["balanced_accuracy_std"],
       capsize=3, label="Balanced Acc", color="#2c7fb8")
ax.bar(x + w/2, summ["macro_f1_mean"], w, yerr=summ["macro_f1_std"],
       capsize=3, label="Macro-F1", color="#31a354")
ax.set_xticks(x); ax.set_xticklabels(summ["experiment"]); ax.set_ylim(0, 1)
ax.set_ylabel("score"); ax.legend(); ax.set_title("Modality comparison (A0 / A1 / A2)")
fig.tight_layout(); save_fig(fig, "fig_modality_comparison")

# ---- leakage comparison bar chart ----
leak = pd.read_csv(os.path.join(PATHS["paper_outputs"], "table_leakage_split.csv"))
def _mean(s): return float(s.split("±")[0])
fig, ax = plt.subplots(figsize=(5.6, 3.6), dpi=300)
x = np.arange(len(leak)); w = 0.38
ax.bar(x - w/2, [_mean(v) for v in leak["patient_split"]], w, label="patient split", color="#2c7fb8")
ax.bar(x + w/2, [_mean(v) for v in leak["random_split"]], w, label="random split", color="#d62728")
ax.set_xticks(x); ax.set_xticklabels(leak["metric"], rotation=20); ax.set_ylim(0, 1)
ax.set_ylabel("score"); ax.legend(); ax.set_title("Leakage: patient vs random split (A2)")
fig.tight_layout(); save_fig(fig, "fig_leakage_comparison")

print("CELL 24 complete — paper figures saved (png + pdf).")

In [ ]:
# =====================================================================
# CELL 25 — FINAL EXPORT + paper_summary.txt

summ = pd.read_csv(os.path.join(PATHS["tables"], "mean_std_summary.csv"))
best_row = summ.loc[summ["macro_f1_mean"].idxmax()]
best_tag = best_row["experiment"]


example_ckpt = f"best_{best_tag}_fold0.pt"
best_model, ck = load_checkpoint(example_ckpt)
n_params = sum(p.numel() for p in best_model.parameters())


assert ck.get("meta_dim", META_DIM) == META_DIM, (
    f"best_tag={best_tag} checkpoint has meta_dim={ck.get('meta_dim')}, "
    f"expected {META_DIM} -- mismatched preprocessing, inference will fail.")


loaders0, _ = make_loaders(folds[0], include_biopsy=False)
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.time(); _ = infer(best_model, loaders0["test"])
torch.cuda.synchronize() if torch.cuda.is_available() else None
n_test = len(loaders0["test"].dataset); infer_ms = (time.time() - t0) / max(1, n_test) * 1000
del best_model; gc.collect(); torch.cuda.empty_cache()


sp = os.path.join(PATHS["paper_outputs"], "paper_summary.txt")
with open(sp, "w") as f:
    f.write("="*70 + "\n")
    f.write("HOW MUCH DOES EACH MODALITY MATTER? — RESULTS SUMMARY\n")
    f.write("PAD-UFES-20 | leakage-controlled comparative study\n")
    f.write("="*70 + "\n\n")
    f.write(f"Dataset: {len(df)} images, {df['_patient_id'].nunique()} patients, "
            f"{CONFIG['num_classes']} classes\n")
    f.write(f"Protocol: patient-grouped stratified {CONFIG['n_folds']}-fold CV; "
            f"biopsy flag + IDs excluded from features\n\n")
    f.write("MAIN RESULTS (mean ± std):\n")
    for _, r in summ.iterrows():
        f.write(f"  {r['experiment']}: BAcc {r['balanced_accuracy_mean']:.3f}"
                f"±{r['balanced_accuracy_std']:.3f} | mF1 {r['macro_f1_mean']:.3f}"
                f"±{r['macro_f1_std']:.3f} | AUROC {r['auroc_mean']:.3f}±{r['auroc_std']:.3f}\n")
    f.write(f"\nBest model: {best_tag} (mean macro-F1 {best_row['macro_f1_mean']:.4f})\n")
    f.write(f"Parameters: {n_params/1e6:.2f} M\n")
    f.write(f"Inference: {infer_ms:.2f} ms/image\n\n")
    for nm, title in [("table_leakage_split.csv", "LEAKAGE — patient vs random split"),
                      ("table_leakage_biopsy.csv", "LEAKAGE — with vs without biopsy flag"),
                      ("table_classwise_modality.csv", "CLASS-WISE MODALITY F1")]:
        p = os.path.join(PATHS["paper_outputs"], nm)
        if os.path.exists(p):
            f.write(title + ":\n" + pd.read_csv(p).to_string(index=False) + "\n\n")
    f.write("LIMITATIONS: single-center, smartphone-only, skewed Fitzpatrick, "
            "n(MEL)=52, simple imputation, concatenation fusion.\n")

# ---- final console block ----
print("="*64)
print(" FINAL RESULTS")
print("="*64)
print(summ.to_string(index=False))
print(f"\n Best model        : {best_tag} (mean macro-F1 {best_row['macro_f1_mean']:.4f})")
print(f" Parameters        : {n_params/1e6:.2f} M")
print(f" Inference         : {infer_ms:.2f} ms/image")
print("\n Saved artifacts:")
for d in SUBDIRS:
    n = len(os.listdir(PATHS[d]))
    print(f"   outputs/{d}/  ({n} files)")
print(f"\n paper_summary.txt -> {sp}")
print("="*64)
print(" Reminder: Save Version -> Save & Run All (Commit) to persist outputs/.")
print("CELL 25 complete.")

In [ ]:
# =====================================================================
# CELL 26 — METABLOCK BASELINE MODEL

class MetaBlockModel(nn.Module):
    def __init__(self, meta_dim, num_classes=6, emb=256, pretrained=True, freeze_early=True):
        super().__init__()
        # image backbone -> (B,1280,7,7) feature map (kept as A for Grad-CAM)
        self.backbone = timm.create_model("efficientnet_b0", pretrained=pretrained,
                                           num_classes=0, global_pool="")
        self.feat_ch = self.backbone.num_features          # 1280
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.img_proj = nn.Sequential(nn.Linear(self.feat_ch, emb), nn.ReLU())

        # metadata branch produces modulation params (gamma, beta) of size `emb`
        self.meta_mlp = nn.Sequential(
            nn.Linear(meta_dim, 128), nn.ReLU(), nn.Dropout(0.3))
        self.to_gamma = nn.Linear(128, emb)                # per-channel scale
        self.to_beta  = nn.Linear(128, emb)                # per-channel shift

        self.classifier = nn.Sequential(
            nn.Linear(emb, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes))

        if freeze_early:
            for p in self.backbone.conv_stem.parameters(): p.requires_grad = False
            for p in self.backbone.bn1.parameters():       p.requires_grad = False
            for i, blk in enumerate(self.backbone.blocks):
                for p in blk.parameters(): p.requires_grad = (i >= 4)

    def forward(self, image, meta):
        A = self.backbone(image)                           # (B,1280,7,7)
        g = self.img_proj(self.gap(A).flatten(1))          # (B,emb) image embedding
        h = self.meta_mlp(meta)                            # (B,128)
        gamma = self.to_gamma(h)                           # (B,emb)
        beta  = self.to_beta(h)                            # (B,emb)
        # MetaBlock modulation: metadata gates the image features
        fused = torch.sigmoid(gamma) * g + beta            # (B,emb)
        logits = self.classifier(fused)
        return {"logits": logits, "A": A}


# sanity check
_m = MetaBlockModel(META_DIM, CONFIG["num_classes"], pretrained=False).eval()
_img = torch.randn(4, 3, CONFIG["image_size"], CONFIG["image_size"])
_meta = torch.randn(4, META_DIM)
with torch.no_grad():
    _o = _m(_img, _meta)
print("MetaBlockModel  logits:", tuple(_o["logits"].shape), "| A:", tuple(_o["A"].shape))
print("  params total/trainable:",
      f"{sum(p.numel() for p in _m.parameters())/1e6:.2f}M /",
      f"{sum(p.numel() for p in _m.parameters() if p.requires_grad)/1e6:.2f}M")
del _m, _o; gc.collect()
print("CELL 26 complete.")

In [ ]:
# =====================================================================
# CELL 27 — REGISTER 'A2mb' IN build_model (non-destructive shim)


_orig_build_model = build_model  

def build_model(name, meta_dim=META_DIM, pretrained=True):
    if name == "A2mb":
        return MetaBlockModel(meta_dim, CONFIG["num_classes"], pretrained=pretrained).to(CONFIG["device"])
    return _orig_build_model(name, meta_dim, pretrained)

# quick check that dispatch works for both old and new tags
_t = build_model("A2mb", META_DIM, pretrained=False); print("A2mb ->", type(_t).__name__)
del _t; gc.collect(); torch.cuda.empty_cache()
print("CELL 27 complete — build_model now dispatches A0/A1/A2/A2mb.")

In [ ]:
# =====================================================================
# CELL 28 — TRAIN MetaBlock BASELINE (A2mb) — patient-level folds


print("########## TRAINING MetaBlock BASELINE (A2mb) ##########")
t0 = time.time()
A2mb_ckpts = run_experiment("A2mb", "A2mb", folds, include_biopsy=False)
print(f"\nA2mb done in {(time.time()-t0)/60:.1f} min. Checkpoints: {A2mb_ckpts}")
print("CELL 28 complete.")

In [ ]:
# =====================================================================
# CELL 29 — EVALUATE A2mb + MERGE INTO all_fold_metrics.csv


print("########## EVALUATING A2mb ##########")
metrics_A2mb = evaluate_experiment("A2mb", folds)

# merge into the in-memory + on-disk master metrics
if "ALL_METRICS" not in dir() or ALL_METRICS is None:
    ALL_METRICS = pd.read_csv(os.path.join(PATHS["metrics"], "all_fold_metrics.csv"))

# drop any stale A2mb rows (idempotent re-run), then append fresh
ALL_METRICS = ALL_METRICS[ALL_METRICS["tag"] != "A2mb"]
ALL_METRICS = pd.concat([ALL_METRICS, metrics_A2mb], ignore_index=True)
ALL_METRICS.to_csv(os.path.join(PATHS["metrics"], "all_fold_metrics.csv"), index=False)

print("\nA2mb per-fold scalar metrics:")
print(metrics_A2mb[["fold", "accuracy", "balanced_accuracy", "macro_f1", "auroc"]].round(4).to_string(index=False))
print("\nMerged into all_fold_metrics.csv. CELL 29 complete.")

In [ ]:
# =====================================================================
# CELL 30 — A2 (ours) vs A2mb (baseline): table + figure + paired test


from scipy.stats import wilcoxon
import matplotlib.pyplot as plt

if "ALL_METRICS" not in dir():
    ALL_METRICS = pd.read_csv(os.path.join(PATHS["metrics"], "all_fold_metrics.csv"))

SCALARS = ["accuracy", "balanced_accuracy", "macro_f1", "auroc"]

# ---- comparison table (include A2 and A2mb; A0/A1 for context) ----
rows = []
for tag, name in [("A0", "Image-only"), ("A1", "Metadata-only"),
                  ("A2", "Concat fusion (ours)"), ("A2mb", "MetaBlock (baseline)")]:
    sub = ALL_METRICS[ALL_METRICS.tag == tag]
    if len(sub) == 0:
        print(f"[warn] no rows for {tag}"); continue
    r = {"model": name, "tag": tag}
    for m in SCALARS:
        r[f"{m}_mean"] = round(sub[m].mean(), 4); r[f"{m}_std"] = round(sub[m].std(), 4)
    rows.append(r)
cmp_df = pd.DataFrame(rows)
cmp_df.to_csv(os.path.join(PATHS["tables"], "baseline_comparison.csv"), index=False)
cmp_df.to_csv(os.path.join(PATHS["paper_outputs"], "table_baseline_comparison.csv"), index=False)

print("===== A2 (ours) vs A2mb (baseline) — mean ± std =====")
for _, r in cmp_df.iterrows():
    print(f" {r['model']:<22} "
          f"Acc {r['accuracy_mean']:.3f}±{r['accuracy_std']:.3f} | "
          f"BAcc {r['balanced_accuracy_mean']:.3f}±{r['balanced_accuracy_std']:.3f} | "
          f"mF1 {r['macro_f1_mean']:.3f}±{r['macro_f1_std']:.3f} | "
          f"AUROC {r['auroc_mean']:.3f}±{r['auroc_std']:.3f}")

# ---- paired A2 vs A2mb across folds ----
def vec(tag, m): return ALL_METRICS[ALL_METRICS.tag == tag].sort_values("fold")[m].values
print("\n===== PAIRED A2 vs A2mb (n=5 folds) =====")
stat_rows = []
for m in ["balanced_accuracy", "macro_f1", "auroc"]:
    a, b = vec("A2", m), vec("A2mb", m)
    if len(a) == len(b) == CONFIG["n_folds"]:
        d = a - b; dz = d.mean() / (d.std(ddof=1) + 1e-12)
        try: _, p = wilcoxon(a, b)
        except ValueError: p = 1.0
        stat_rows.append({"metric": m, "mean_diff_A2_minus_A2mb": round(d.mean(), 4),
                          "cohen_dz": round(dz, 2), "wilcoxon_p": round(p, 3)})
        print(f"  {m}: Δ(A2−A2mb)={d.mean():+.4f}  d_z={dz:+.2f}  p={p:.3f}")
pd.DataFrame(stat_rows).to_csv(os.path.join(PATHS["paper_outputs"], "table_baseline_paired.csv"), index=False)
print("  (n=5 -> p floor is 0.062; read effect size + sign, not significance.)")

# ---- comparison bar figure (BAcc + macro-F1) ----
fig, ax = plt.subplots(figsize=(6.4, 3.8), dpi=300)
x = np.arange(len(cmp_df)); w = 0.38
ax.bar(x - w/2, cmp_df["balanced_accuracy_mean"], w, yerr=cmp_df["balanced_accuracy_std"],
       capsize=3, label="Balanced Acc", color="#2c7fb8")
ax.bar(x + w/2, cmp_df["macro_f1_mean"], w, yerr=cmp_df["macro_f1_std"],
       capsize=3, label="Macro-F1", color="#31a354")
ax.set_xticks(x); ax.set_xticklabels(cmp_df["model"], rotation=20, ha="right", fontsize=8)
ax.set_ylim(0, 1); ax.set_ylabel("score"); ax.legend()
ax.set_title("Modality baselines vs MetaBlock (patient-grouped 5-fold)")
fig.tight_layout()
for ext in ["png", "pdf"]:
    fig.savefig(os.path.join(PATHS["figures"], f"fig_baseline_comparison.{ext}"), bbox_inches="tight")
plt.close(fig)
print("\nSaved baseline_comparison.csv, table_baseline_paired.csv, fig_baseline_comparison.png/.pdf")
print("CELL 30 complete.")

In [ ]:


import os, glob
import pandas as pd
from IPython.display import display, Image as IPImage, Markdown

OUT = "/kaggle/working/outputs"

def show_header(txt):
    display(Markdown(f"## {txt}"))

# ---------- 1. ALL TABLES (CSV -> rendered dataframes) ----------
show_header("📊 TABLES")
table_dirs = ["paper_outputs", "tables", "metrics", "metadata_analysis"]
seen = set()
for d in table_dirs:
    for csv in sorted(glob.glob(os.path.join(OUT, d, "*.csv"))):
        name = os.path.basename(csv)
        if name in seen:            # avoid duplicates saved in two places
            continue
        seen.add(name)
        try:
            df_ = pd.read_csv(csv)
            display(Markdown(f"**{d}/{name}**  ({len(df_)} rows)"))
            display(df_)
        except Exception as e:
            print(f"[skip] {name}: {e}")

# ---------- 2. ALL FIGURES (PNG) ----------
show_header("📈 FIGURES")
fig_pngs = sorted(glob.glob(os.path.join(OUT, "figures", "*.png")))
for png in fig_pngs:
    display(Markdown(f"**figures/{os.path.basename(png)}**"))
    display(IPImage(filename=png))

# ---------- 3. GRAD-CAM IMAGES ----------
show_header("🔬 GRAD-CAM")
cam_pngs = sorted(glob.glob(os.path.join(OUT, "gradcam", "*.png")))
for png in cam_pngs:
    display(Markdown(f"**gradcam/{os.path.basename(png)}**"))
    display(IPImage(filename=png))

# ---------- 4. PAPER SUMMARY TEXT ----------
show_header("📄 PAPER SUMMARY")
summ_path = os.path.join(OUT, "paper_outputs", "paper_summary.txt")
if os.path.exists(summ_path):
    with open(summ_path) as f:
        display(Markdown("```\n" + f.read() + "\n```"))

# ---------- 5. INVENTORY (what got saved) ----------
show_header("🗂️ FULL OUTPUT INVENTORY")
for root, dirs, files in os.walk(OUT):
    rel = os.path.relpath(root, OUT)
    if files:
        print(f"\n{rel}/ ({len(files)} files)")
        for fn in sorted(files):
            print(f"    {fn}")
print("\nCELL 31 complete — all publication outputs displayed inline above.")